# 实验 2：优化器的更新几何与有限步 polar 近似

**对应第4讲；课堂实验约1.5学时。** 本 Notebook 调用 `experiments/lab02_optimizers.py`。默认小网络、一个随机种子先运行；后面提供默认关闭的三种子扩展。

比较对象是脚本明确定义的 `SGD+momentum`、`AdamW` 与 `Muon+AdamW`。同一随机种子内，它们共享数据划分、模型初始化、批次索引序列和训练步数。每种方法使用预先声明的三个学习率候选，再按最终验证损失选择；测试集只用于选择后的评价。

**单种子、有限学习率扫描不能建立某优化器普遍优于其他方法的结论。** 本实验首先检验更新长度、下降方向对齐及数值近似，而非预设某一种方法必须胜出。


## 环境与执行方式

在已有的 Jupyter 或 VS Code Notebook 环境中，选择安装了课程依赖的 Python 内核，按顺序运行单元格。计算依赖与命令行脚本相同；展示使用 Notebook 环境已有的 `IPython.display`，不需要 pandas 或额外的交互组件。

如尚未安装依赖，请在**课程目录的终端**手动运行：

```bash
python -m pip install -r requirements.txt
```

本 Notebook 不会自动安装包，也不会下载外部数据集。生成的图、CSV 和 JSON 写入课程目录下的 `results/notebooks/`，不会写入 Notebook 的源码单元格。仓库版本有意保留空输出；运行后可以通过生成文件复核结果。


## 课前：先明确你比较的是哪个算法

1. 谱范数球内线性最小化为什么给出 $-\rho U_rV_r^\top$？秩亏时为什么不能任意补齐零奇异方向？
2. Adam 什么时候严格化为 SignGD？有动量时，更新是否必须与当前负梯度同向？
3. 未乘学习率的更新 RMS、实际参数位移 RMS、相对更新长度是三个什么量？
4. 为什么 AdamW 的解耦衰减与 SGD 的耦合 L2 在非零衰减设置下不是相同操作？

课堂默认 `weight_decay=0`，先隔离更新几何。Muon 只更新前两层的二维权重；bias 和最后的 readout 由 AdamW 更新，辅助学习率为矩阵学习率的十分之一。Muon 使用 EMA 动量系数0.95、三次 Newton–Schulz、Frobenius 初始化，以及 $0.2\sqrt{\max(m,n)}$ 的形状缩放。


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import json
import sys
from IPython.display import display, Image, JSON, Markdown

# 可从 notebooks/、课程目录或仓库根目录启动，不依赖机器上的绝对路径。
start = Path.cwd().resolve()
candidates = []
for parent in (start, *start.parents):
    candidates.extend((parent, parent / '2027' / 'nonlinear optimization'))
COURSE_ROOT = next(
    (p for p in candidates
     if (p / 'experiments' / 'lab01_sgd.py').is_file()
     and (p / 'chapters').is_dir()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError('未找到课程目录：请从本仓库的课程目录或 notebooks 目录启动。')
EXPERIMENTS = COURSE_ROOT / 'experiments'
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
print('课程目录：', COURSE_ROOT)
print('实验实现：', EXPERIMENTS)


In [ ]:
from lab02_optimizers import Config as LabConfig, run as run_lab

# 改成 device='cpu' 可固定CPU；'cuda' 要求本机PyTorch确有可用CUDA。
config = LabConfig(
    seed=2027, steps=80, batch=64, width=16,
    device='auto', polar_method='ns', ns_steps=5, weight_decay=0.0,
)
OUTPUT = COURSE_ROOT / 'results' / 'notebooks' / 'lab02' / f'seed_{config.seed}'
display(JSON({'config': asdict(config), 'output': str(OUTPUT)}))


## 课中 1：运行共同初始化与候选网格

为快速交互，默认宽度16、训练80步；命令行脚本默认宽度32、训练200步。结果必须连同这些设置报告。

预声明候选为：

| 方法 | 候选学习率 | 补充约定 |
|---|---|---|
| SGD+momentum | 0.01、0.03、0.1 | momentum=0.9 |
| AdamW | 0.001、0.003、0.01 | 使用脚本中的 PyTorch AdamW 默认矩参数 |
| Muon+AdamW | 0.01、0.03、0.1 | 辅助 AdamW 学习率为候选值/10 |

这里的“最佳”始终指该种子、数据和预算下，在这三个候选中的验证选择。运行时间包含诊断和评估开销，不是硬件性能基准。


In [ ]:
report = run_lab(config, output=OUTPUT)
display(Image(filename=str(OUTPUT / 'lab02_optimizers.png')))
display(Image(filename=str(OUTPUT / 'polar_approximation.png')))


In [ ]:
saved_report = json.loads((OUTPUT / 'report.json').read_text(encoding='utf-8'))
display(JSON({
    'actual_device': saved_report['actual_device'],
    'config': saved_report['config'],
    'selected': saved_report['selected'],
    'protocol': saved_report['protocol'],
}))
display(JSON(saved_report['candidates']))


## 课中 2：从数值记录解释方向

主图的相对更新是 $\|\Delta\theta\|/\|\theta\|$；CSV 中 `update_rms` 是包含学习率的**实际参数位移**，而且聚合了所有参数，包含 Muon 的辅助 AdamW 参数。它不是理论公式中仅对某个矩阵、未乘学习率的 polar RMS。

`descent_cosine` 使用当次批次梯度与实际位移：

$$-\frac{\langle g,\Delta\theta\rangle}{\|g\|\|\Delta\theta\|}.$$

接近1表示当前下降方向对齐较强，不表示验证集性能必然更好；动量的历史记忆也可能使这个量暂时很小或为负。


In [ ]:
import csv

with (OUTPUT / 'all_curves.csv').open(encoding='utf-8', newline='') as handle:
    curve_rows = list(csv.DictReader(handle))
last_record = {}
for row in curve_rows:
    last_record[row['candidate']] = row
chosen_last_records = {
    name: last_record[info['candidate']]
    for name, info in saved_report['selected'].items()
}
display(JSON(chosen_last_records))
display(JSON(saved_report['polar_example']))


## 课中 3：可选的 SVD 训练对照

上面的奇异值图已经比较了精确 SVD polar 与三次 NS 的有限步结果。下面进一步比较训练，把 `RUN_SVD_TRAINING` 改为 `True` 可执行。其余配置保持不变，结果写入单独目录。

SVD 版本是方向参考，不要求它在时间上更快。三次NS在小奇异值处的有限步行为与精确polar不同，不能把结果差异全部归因于“同一算法的计算加速”。


In [ ]:
RUN_SVD_TRAINING = False
svd_report = None
if RUN_SVD_TRAINING:
    svd_config = replace(config, polar_method='svd')
    svd_output = COURSE_ROOT / 'results' / 'notebooks' / 'lab02' / f'svd_seed_{config.seed}'
    svd_report = run_lab(svd_config, output=svd_output)
    display(Image(filename=str(svd_output / 'lab02_optimizers.png')))
    display(JSON({'ns_selected': report['selected'], 'svd_selected': svd_report['selected']}))
else:
    print('未启动额外训练；将 RUN_SVD_TRAINING 改为 True 后重跑本格，可与SVD参考比较。')


## 课后扩展：至少三个随机种子

先保留共同的配置和候选网格，再打开下面的开关。三个种子使用独立输出目录，不覆盖单种子演示。

当前脚本中的 `seed` 同时影响合成数据、初始化和批次流。因此这组三种子结果描述的是这些随机性的共同变化；它不是固定数据、仅改变初始化的隔离实验。每个种子内部仍保持各方法之间的共同协议。


In [ ]:
RUN_MULTIPLE_SEEDS = False
SEEDS = (2027, 2028, 2029)
multi_reports = {}
if RUN_MULTIPLE_SEEDS:
    for seed in SEEDS:
        seeded_config = replace(config, seed=seed)
        seeded_output = COURSE_ROOT / 'results' / 'notebooks' / 'lab02' / f'multi_seed_{seed}'
        multi_reports[seed] = run_lab(seeded_config, output=seeded_output)
        print('已完成种子', seed, '设备', multi_reports[seed]['actual_device'])
else:
    print('三种子扩展默认关闭；当前结果仅是单种子演示，不能据此比较普遍优劣。')


In [ ]:
from statistics import mean, stdev

if multi_reports:
    selected_rows = [
        {'seed': seed, 'optimizer': name, **info}
        for seed, item in multi_reports.items()
        for name, info in item['selected'].items()
    ]
    aggregated = []
    for name in sorted({row['optimizer'] for row in selected_rows}):
        rows = [row for row in selected_rows if row['optimizer'] == name]
        values = [row['test_loss'] for row in rows]
        aggregated.append({
            'optimizer': name,
            'number_of_seeds': len(values),
            'mean_selected_test_loss': mean(values),
            'sample_standard_deviation': stdev(values) if len(values) > 1 else None,
            'selected_candidates': [row['candidate'] for row in rows],
        })
    display(JSON({'per_seed_validation_selected_results': selected_rows, 'descriptive_summary': aggregated}))
else:
    print('尚未产生三种子统计；启用上一格后重跑本格。均值和标准差只作描述，不自动构成显著性结论。')


## 提交问题与核验要点

1. 按相同预算报告候选网格、验证选择及每个种子的配置。若最优候选落在网格端点，为什么仍不能说找到了最优学习率？
2. 比较实际位移、相对更新和下降方向对齐。哪一种量解释了你观察到的训练差异，哪一种解释仍需额外对照？
3. NS5 对小奇异值的处理与SVD有何不同？说明该差异可能同时影响数值成本与训练机制。
4. 为什么相同RMS不保证相同损失进展？给出一个与梯度正交但长度相同的更新反例。
5. 若打开非零权重衰减，为什么这已不是只改变几何的单因素比较？

**核验标准：** 不预设方法排序；保留失败配置及预算信息；不使用测试集挑学习率；明确混合优化器的参数分工。三种子和有限网格仍只支持当前任务与配置范围内的描述性结论。
